# Train Test Split

## Build the Complete Feature Schema

- **Purpose:** Build the complete point-in-time event feature schema from news and market features.
- **Settings:** Deduplicate news by time/headline; align to the first bar end ≥ publication; average sentiment per bar; exact-time left joins; February–December 2025.
- **Data:** News sentiment, one fractional-price feature, and 48 technical features produce the 50-feature event schema.
- **Findings:**
- **Decision:** Preserve missing values for the later cleaning stage.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.preprocessing.market_data import ResearchPaths
from src.preprocessing.train_test_split import build_research_candidates
from src.preprocessing.market_technical_indicators import MODEL_FEATURES
from src.preprocessing.train_test_split import chronological_train_test_split

paths = ResearchPaths(PROJECT_ROOT)
candidate_path = paths.event("candidates")
event_dir = candidate_path.parent
holdout_boundary = None
if candidate_path.exists():
    stored = pd.read_parquet(candidate_path, columns=["holdout_boundary"])
    if stored.holdout_boundary.nunique() != 1:
        raise ValueError("Stored candidates must have one immutable boundary")
    holdout_boundary = stored.holdout_boundary.iloc[0]


In [ ]:
candidate_schema = build_research_candidates(paths)
feature_columns = list(MODEL_FEATURES)
display(pd.DataFrame(candidate_schema.attrs.get("excluded_symbols", [])))


## Create a Test Set

- **Purpose:** Apply the immutable chronological development and holdout partition.
- **Settings:** Target development/holdout = 80/20 pooled rows; keep equal timestamps together and reuse the frozen UTC boundary.
- **Data:** Persist the full feature schema, including missing values, with partition and holdout-boundary metadata.
- **Findings:**
- **Decision:** Assign earlier rows to development and rows at or after the fixed boundary to sealed holdout.

In [ ]:
development, holdout, manifest = chronological_train_test_split(
    candidate_schema,
    holdout_boundary=holdout_boundary,
)
candidate_split = candidate_schema.merge(manifest, on=["symbol", "event_start"], how="left", validate="one_to_one")
candidate_split = candidate_split[["event_start", "symbol", "partition", "holdout_boundary", *feature_columns]]

event_dir.mkdir(parents=True, exist_ok=True)
candidate_split.to_parquet(candidate_path, index=False)
print(candidate_path)

holdout_boundary = manifest.holdout_boundary.iloc[0]


In [ ]:
partition_summary = pd.DataFrame(
    {
        "events": [len(development), len(holdout)],
        "start": [development["event_start"].min(), holdout["event_start"].min()],
        "end": [development["event_start"].max(), holdout["event_start"].max()],
    },
    index=pd.Index(["development", "holdout"], name="partition"),
)
display(partition_summary)
print(f"holdout_boundary: {holdout_boundary}")